# PISA 2025 – prvé zoznámenie s dátami

Tento notebook je východiskový bod projektu. Najprv si prezrieme dodaný
žiacky súbor, jeho premenné a malú ukážku záznamov. Jeden riadok predstavuje
záznam žiaka; stĺpce zahŕňajú identifikátory, odpovede a odvodené premenné.

Zatiaľ nevyberáme výskumnú otázku ani netrénujeme modely. Zdrojové súbory
nemeníme. Bunky spúšťaj postupne zhora nadol.


## 1. Kde sú dáta

Použijeme pôvodný SPSS súbor v `data/raw/`. Je veľký približne 2 GB,
preto budeme najprv čítať len jeho popis a malú ukážku.
Codebook s významom premenných je v `data/reference/PISA2025_Codebook.xlsx`.

Nasledujúca bunka funguje pri spustení z hlavného priečinka projektu
aj z priečinka `notebooks/`.


In [1]:
from pathlib import Path
import pandas as pd
import pyreadstat

PROJECT = Path.cwd()
if not (PROJECT / "data" / "raw").is_dir():
    PROJECT = PROJECT.parent

DATA = PROJECT / "data" / "raw" / "CY09_MS_STU_PUF.sav"
if not DATA.is_file():
    raise FileNotFoundError(f"Chýba pôvodný dátový súbor: {DATA}")

print(f"Dátový súbor: {DATA.name}")

Dátový súbor: CY09_MS_STU_PUF.sav


## 2. Koľko údajov obsahuje súbor

Metadáta sú popis súboru: počet riadkov, názvy premenných a ich významy.
Voľba `metadataonly=True` načíta tento popis bez celej tabuľky žiakov.


In [2]:
_, meta = pyreadstat.read_sav(DATA, metadataonly=True)

pd.Series({
    "Počet záznamov žiakov": meta.number_rows,
    "Počet premenných": meta.number_columns,
}, name="hodnota")

Počet záznamov žiakov    755721
Počet premenných            885
Name: hodnota, dtype: int64

## 3. Aké premenné máme k dispozícii

Vytvoríme prehľad názvov, popisov a pôvodných dátových typov. Popisy sú
prevzaté zo SPSS súboru a zostávajú v pôvodnom jazyku. Zobrazíme prvých
20 premenných; celý prehľad je uložený v premennej `variables`.


In [3]:
variables = pd.DataFrame({
    "premenná": meta.column_names,
    "popis": [meta.column_names_to_labels.get(name, "") for name in meta.column_names],
    "typ": [meta.original_variable_types.get(name, "") for name in meta.column_names],
})

variables.head(20)

,premenná,popis,typ
0,CNT,Country code 3-character,A3
1,CNTRYID,3-digit country code,A3
2,CNTSCHID,Intl. School ID,A8
3,CNTSTUID,Intl. Student ID,A8
4,CYC,PISA Assessment Cycle (2 digits + 2 character ...,A4
5,NatCen,National Centre 6-digit Code,A6
6,STRATUM,Stratum ID 5-character (cnt + original stratum...,A5
7,SUBNATIO,Adjudicated sub-region code 7-digit code (3-di...,A7
8,REGION,REGION,A5
9,OECD,OECD country,F1.0


### Vyhľadanie podľa názvu alebo významu

Zmeň text v `SEARCH`. Napríklad `gender`, `belong`, `age` alebo `ESCS`.
Vyhľadávame v názve aj popise premenných bez rozlišovania veľkých písmen.


In [ ]:
SEARCH = "belong"

matches = variables["premenná"].str.contains(SEARCH, case=False, regex=False)
matches |= variables["popis"].str.contains(SEARCH, case=False, regex=False, na=False)
variables.loc[matches]

## 4. Ako vyzerajú jednotlivé záznamy

Načítame prvých **20 riadkov** a iba šesť premenných: krajinu, identifikátory
školy a žiaka, pohlavie, vek a index ekonomického, sociálneho a kultúrneho
zázemia (`ESCS`). Ide o ukážku štruktúry. Prvých 20 riadkov nie je náhodný
ani reprezentatívny výber.

`apply_value_formats=False` ponechá pôvodné kódy odpovedí, aby sme videli,
čo je v súbore skutočne uložené.


In [4]:
preview_columns = ["CNT", "CNTSCHID", "CNTSTUID", "ST004D01T", "AGE", "ESCS"]

preview, _ = pyreadstat.read_sav(
    DATA,
    usecols=preview_columns,
    row_limit=20,
    apply_value_formats=False,
)
preview = preview[preview_columns]
preview

,CNT,CNTSCHID,CNTSTUID,ST004D01T,AGE,ESCS
0,ALB,00800001,00800001,2.0,15.83,-1.100674
1,ALB,00800001,00800002,1.0,15.92,-1.930787
2,ALB,00800001,00800003,1.0,15.83,-1.078518
3,ALB,00800001,00800004,1.0,16.25,-0.018204
4,ALB,00800001,00800005,1.0,15.50,-1.071648
5,ALB,00800001,00800006,2.0,16.17,-0.085581
6,ALB,00800001,00800007,2.0,16.00,NaN
7,ALB,00800001,00800008,2.0,15.83,-1.236167
8,ALB,00800001,00800009,2.0,15.33,-2.176807
9,ALB,00800001,00800010,2.0,15.58,-0.735643


### Čo znamenajú číselné kódy

Číslo môže byť označenie kategórie. Napríklad význam kódov pohlavia
v `ST004D01T` zistíme z popisov vložených do súboru. Na interpretáciu
ďalších premenných máme aj dodaný codebook.


In [5]:
VARIABLE = "ST004D01T"

pd.DataFrame.from_dict(
    meta.variable_value_labels.get(VARIABLE, {}),
    orient="index",
    columns=["význam"],
).rename_axis("kód")

,význam
kód,
1.0,Female
2.0,Male
95.0,valid skip (filter rule)
96.0,system missing
97.0,not administered/not applicable
98.0,invalid
99.0,no response


## 5. Chýbajúce hodnoty v malej ukážke

`isna()` spočíta prázdne hodnoty (`NaN`) v načítaných 20 riadkoch.
Tieto počty opisujú len ukážku, nie celý dataset.

Niektoré premenné majú navyše osobitné kódy pre neodpovedanie, preskočenú
otázku alebo neplatnú odpoveď. Ich význam závisí od konkrétnej premennej;
v ukážke kódov vyššie sú napríklad hodnoty 95 až 99. Nasledujúci prehľad
takéto číselné kódy nepočíta ako `NaN`. Zatiaľ žiadne hodnoty neprepisujeme
a nepoužívame jeden spoločný zoznam chýbajúcich kódov pre všetky stĺpce.


In [6]:
pd.DataFrame({
    "počet NaN v ukážke": preview.isna().sum(),
    "podiel NaN v ukážke (%)": preview.isna().mean().mul(100).round(1),
})

,počet NaN v ukážke,podiel NaN v ukážke (%)
CNT,0,0.0
CNTSCHID,0,0.0
CNTSTUID,0,0.0
ST004D01T,0,0.0
AGE,0,0.0
ESCS,3,15.0


## 6. Ktoré krajiny a ekonomiky sú v súbore

Teraz načítame **iba stĺpec `CNT`, ale pre všetky riadky**. Táto časť musí
prejsť veľký SPSS súbor a môže chvíľu trvať. Do pamäte sa načíta len jeden
stĺpec, nie všetkých 885 premenných.

Výsledkom budú počty záznamov v súbore podľa krajiny alebo ekonomiky.
Sú to nevážené počty pozorovaní, ktoré používame iba na orientáciu v dátach.


In [ ]:
countries, _ = pyreadstat.read_sav(
    DATA,
    usecols=["CNT"],
    apply_value_formats=False,
)

country_counts = countries["CNT"].value_counts(dropna=False)
country_overview = country_counts.rename_axis("kód").to_frame("počet záznamov")
country_overview.insert(0, "krajina / ekonomika", country_overview.index.map(meta.variable_value_labels["CNT"]))

print(f"Počet neprázdnych kódov krajín / ekonomík: {countries['CNT'].nunique()}")
print(f"Počet načítaných záznamov: {len(countries):,}")
country_overview

## Ďalší krok

Po prejdení notebooku sa rozhodneme, čo chceme skúmať. Až potom vyberieme
krajiny, premenné, spôsob práce s chýbajúcimi údajmi a vhodnú metódu.

Pri budúcich populačných odhadoch bude potrebné pracovať s výberovými
a replikačnými váhami; pri výsledkoch testov aj s plausible values.
Pôvodné pomocné skripty a podklady sú v `archive/`.
